In [145]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/')
import features.ptype_prepare_data as pp
import models.ptype_run_preds as rp
import models.score_classifier as score
import report.ptype_visualize as viz
import pandas as pd
import pickle
import os
import boto3
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import hydra
from omegaconf import DictConfig
from utils.logs import get_logger
import yaml
from data.s3_download_sso import data_download

In [135]:
config_path='../params.yaml'
with open(config_path) as conf_file:
    config = yaml.safe_load(conf_file)
logger = get_logger('DATA_LOAD', log_level=config['base']['log_level'])

<Logger DATA_LOAD (INFO)>

In [132]:
@hydra.main(config_path='params.yaml')
def main(config: DictConfig) -> None:
    print(config.data_load.sso_profile_name)

In [138]:
config['data_load']['data_prefix_list']

['slope', 's1', 's2', 'dem', 'features-ckpt-2023-02-09', 'labels']

In [146]:
data_download(config_path)

2023-10-16 10:37:57,251 — DATA_DOWNLOAD — INFO — No new data downloaded


In [115]:
ceo_summary = pd.read_excel('../data/Training_Data.xlsx', sheet_name='CEO_Surveys', header=1)
ceo_summary = ceo_summary[ceo_summary['CEO Survey'].notna()]
ceo_summary = ceo_summary.map(lambda x: x.lower() if isinstance(x, str) else x)
ceo_summary

,CEO Survey,Year Collected,Data Source,Municipality,Country,Purpose,Vegetation categories (observed),Plots in Polygons,Plots outside Polygons,Final plot count,Classes,Non plantation (0),Plantation (for mult-class this is mono) (1),Agroforestry (2),Plot ID,Use?,Notes
0,plantations-train-v00,NaN,sdpt,colon,honduras,prototyping,NaN,7.0,0.0,NaN,binary,NaN,NaN,NaN,NaN,yes,NaN
1,plantations-train-v01,NaN,sdpt,colon,honduras,prototyping,NaN,8.0,0.0,NaN,binary,NaN,NaN,NaN,NaN,yes,NaN
2,plantations-train-v02,NaN,sdpt,copan,honduras,prototyping,coffee agroforestry,100.0,0.0,NaN,binary,NaN,NaN,NaN,NaN,no,NaN
3,plantations-train-v03,NaN,sdpt,multiple,guatemala,pilot #2,"oil palm, orchards",100.0,0.0,100.0,binary,NaN,NaN,NaN,0-99,yes,NaN
4,plantations-train-v04,NaN,sdpt,multiple,guatemala,pilot #2,non plantation,0.0,66.0,66.0,binary,NaN,NaN,NaN,4000-40099 (not consecutive),yes,NaN
5,plantations-train-v05,NaN,iiasa,multiple,"brazil, costa rica, ivory coast, ghana, guatemala",prototyping,short rotation plantations for timber,NaN,100.0,NaN,binary,NaN,NaN,NaN,NaN,no,NaN
6,plantations-train-v06,NaN,iiasa,multiple,"brazil, costa rica, ivory coast, ghana, guatem...",prototyping,oil palm,NaN,101.0,101.0,binary,NaN,NaN,NaN,40100-40200,yes,NaN
7,plantations-train-v07,NaN,iiasa,multiple,"brazil, costa rica, ivory coast, ghana, guatem...",prototyping,agroforestry,NaN,101.0,NaN,binary,NaN,NaN,NaN,NaN,no,NaN
8,plantations-train-v08_old,NaN,sdpt,multiple,"ghana, ivory coast",pilot #1,"oil palm, rubber, cashew, agroforestry",100.0,125.0,213.0,binary,25095.0,16653.0,NaN,8001-8225 (not consecutive),yes,agroforestry is labeled as 0
9,plantations-train-v08,NaN,sdpt,multiple,"ghana, ivory coast",pilot #1,"oil palm, rubber, cashew, agroforestry",100.0,125.0,177.0,multi,9179.0,12247.0,13266.0,8000,yes,NaN


In [114]:
ceo_summary_dict = ceo_summary.set_index('CEO Survey').T.to_dict()
ceo_summary_dict.keys()

dict_keys(['plantations-train-v00', 'plantations-train-v01', 'plantations-train-v02', 'plantations-train-v03', 'plantations-train-v04', 'plantations-train-v05', 'plantations-train-v06', 'plantations-train-v07', 'plantations-train-v08_old', 'plantations-train-v08', 'plantations-train-v09', 'plantations-train-v10', 'plantations-train-v11', 'plantations-train-v12', 'plantations-train-v13', 'plantations-train-v14', 'plantations-train-v15', 'plantations-train-v16', 'plantations-train-v17', 'plantations-train-v18', 'plantations-train-v19', 'plantations-train-v20', 'plantations-train-v21', 'plantations-train-v22'])

In [ ]:
def split_dict(item, split_string):
    

In [120]:
ceo_summary_dict['plantations-train-v05']['Country'].split(", ")

['brazil', 'costa rica', 'ivory coast', 'ghana', 'guatemala']

In [9]:
X, y = pp.create_xy(['v08'], 
                    classes='binary', 
                    drop_feats=False,
                    data_dir ='../data', 
                    verbose=False)


0.0 plots labeled unknown were dropped from v08.
warning needs to be updated
Plot id 08001 has no cloud free imagery and will be removed.
Plot id 08002 has no cloud free imagery and will be removed.
Plot id 08003 has no cloud free imagery and will be removed.
Plot id 08004 has no cloud free imagery and will be removed.
Plot id 08005 has no cloud free imagery and will be removed.
Plot id 08006 has no cloud free imagery and will be removed.
Plot id 08007 has no cloud free imagery and will be removed.
Plot id 08008 has no cloud free imagery and will be removed.
Plot id 08009 has no cloud free imagery and will be removed.
Plot id 08010 has no cloud free imagery and will be removed.
Plot id 08011 has no cloud free imagery and will be removed.
Plot id 08012 has no cloud free imagery and will be removed.
Plot id 08013 has no cloud free imagery and will be removed.
Plot id 08014 has no cloud free imagery and will be removed.
Plot id 08015 has no cloud free imagery and will be removed.
Plot id 

0it [00:00, ?it/s]

Class count {}
